<a href="https://colab.research.google.com/github/Kamalashrinithi19/kamalashrinithi-codeboosters-2026/blob/main/Day3/Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install requests --quiet #
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('All libraries impoerted successfully!')
print(f'pandas: {pd.__version__}')
print(f'requests: {requests.__version__}')

All libraries impoerted successfully!
pandas: 2.2.2
requests: 2.32.4


In [8]:
raw_df = pd.read_csv('messy_sales_data.csv')

print(f'Raw data loaded: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
print(f'Columns: {raw_df.columns.tolist()}')
print('\nFirst 5 rows')
print(raw_df.head().to_string(index=False))

Raw data loaded: 30 rows, 9 columns
Columns: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

First 5 rows
 order_id customer_name  product    category  quantity  unit_price order_date      city   sales_rep
     1001  Ramesh Kumar   Laptop Electronics       2.0       45000 2024-01-05    Mumbai Anil Sharma
     1002    Priya Nair      NaN Electronics       1.0       15000 2024-01-07     Delhi  Sunita Rao
     1003    AMIT VERMA Keyboard Accessories       3.0        1200 2024-01-08 Bangalore Anil Sharma
     1004  Sunita Patel  Monitor Electronics       NaN       22000 2024-01-10   Chennai  Ravi Kumar
     1005  Ramesh Kumar   Laptop Electronics       2.0       45000 2024-01-05    Mumbai Anil Sharma


In [10]:
print('='*55)
print('DATA QUALITYDIAGNOSIS REPORT')
print('='*55)

#1. missing values
print('\n[1] MISSING VALUES per column')
print(raw_df.isnull().sum())

#2. Duplicates
print(f'\n[2] DUPLICATES: {raw_df.duplicated().sum()}')

#3. Data Types
print('\n[3] DATA TYPES')
print(raw_df.dtypes)

#4. Unique values in text columns(spot inconsistencies)
print('\n[4] UNIQUE CATEGORIES:', raw_df['category'].unique())
print('[4] Sample customer names:', raw_df['customer_name'].dropna().unique()[:8])
print('[4] Sample order_date values:', raw_df['order_date'].dropna().unique()[:8])

DATA QUALITYDIAGNOSIS REPORT

[1] MISSING VALUES per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATES: 0

[3] DATA TYPES
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12' '2024-01-13' '2024-01-15']


In [11]:
#1. missing values
print('\n[1] MISSING VALUES per quantity')
print(raw_df['quantity'].isnull().sum())


[1] MISSING VALUES per quantity
3


In [12]:
#Create a working copy (ETL best practice)

df=raw_df.copy()
print(f'Working copy created: {df.shape}')
print('raw_df is untouched - we can always reset by running df=raw_df.copy()')

Working copy created: (30, 9)
raw_df is untouched - we can always reset by running df=raw_df.copy()


In [19]:
# 1. Handle missing values

print('Before fixing nulls', df.isnull().sum().sum(), 'total missing values')

df['customer_name'].fillna('Unknown Customer', inplace=True)
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f'Filled missing quantity with median: {median_qty}')

df['category'].fillna('Uncategorized', inplace=True)

print('After fixing nulls:', df.isnull().sum().sum(), 'total missing values')

Before fixing nulls 1 total missing values
Filled missing quantity with median: 2.0
After fixing nulls: 1 total missing values


In [27]:
#2. Remove duplicates

print(f'Before deduplication: {len(df)} rows')
print(f'Duplicate rows: {df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']].head(6))

df.drop_duplicates(inplace=True)

print(f'\nAfter deduplication: {len(df)} rows')
print(f'Rows removed: {len(raw_df)-len(df)}')

Before deduplication: 30 rows
Duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

After deduplication: 30 rows
Rows removed: 0


In [26]:
df['product'].fillna('Unknown product', inplace=True)
print(df['product'].isnull().sum())

0


In [33]:
dup1=df.head(2)
print(dup1)

   order_id customer_name          product     category  quantity  unit_price  \
0      1001  Ramesh Kumar           Laptop  Electronics       2.0       45000   
1      1002    Priya Nair  Unknown product  Electronics       1.0       15000   

   order_date    city    sales_rep  
0  2024-01-05  Mumbai  Anil Sharma  
1  2024-01-07   Delhi   Sunita Rao  


In [37]:
df=pd.concat([df, dup1], ignore_index=True)

In [31]:
print(df.tail(2))

    order_id customer_name          product     category  quantity  \
30      1001  Ramesh Kumar           Laptop  Electronics       2.0   
31      1002    Priya Nair  Unknown product  Electronics       1.0   

    unit_price  order_date    city    sales_rep  
30       45000  2024-01-05  Mumbai  Anil Sharma  
31       15000  2024-01-07   Delhi   Sunita Rao  


In [32]:
print(df.duplicated().sum())

2


In [38]:
n_dup=df.duplicated().sum()
print(f'Duplicated rows: {n_dup}')
print(df[df.duplicated(keep=False)][['order_id', 'customer_name', 'product']].head(6))

df_len_before = len(df)
df.drop_duplicates(inplace=True)
df_len_after = len(df)
print(f'After deduplication: {df_len_after} rows')
print(f'Rows removed: {df_len_before - df_len_after}')

Duplicated rows: 2
    order_id customer_name          product
0       1001  Ramesh Kumar           Laptop
1       1002    Priya Nair  Unknown product
30      1001  Ramesh Kumar           Laptop
31      1002    Priya Nair  Unknown product
After deduplication: 30 rows
Rows removed: 2


In [42]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())

df['order_date']=pd.to_datetime(df['order_date'], dayfirst=False, errors='coerce')

nat_count=df['order_date'].isnull().sum()
print(f'\nUnparseable dates (NaT): {nat_count}')

df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')

print('\nSample dates after parsing:')
print(df[['order_date', 'year', 'month', 'month_name']].head(8))

Sample dates before parsing:
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-08 00:00:00'), Timestamp('2024-01-10 00:00:00'), Timestamp('2024-01-05 00:00:00'), NaT, Timestamp('2024-01-12 00:00:00'), Timestamp('2024-01-13 00:00:00')]

Unparseable dates (NaT): 2

Sample dates after parsing:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January
5        NaT     NaN    NaN        NaN
6 2024-01-12  2024.0    1.0    January
7 2024-01-13  2024.0    1.0    January
